# Notebook 00b — Clean Extraction

**Purpose:** Extract one furnace's data from Databricks, apply all cleaning cuts,
and save a clean parquet file that all downstream notebooks read from.

## What this notebook does

```
Raw Databricks data
  → detect cracking runs (via feed drop)
  → remove: before first complete cycle
  → remove: decoking periods (feed ≈ 0)
  → remove: warm-up at run start (feed stepping up, COT unstable)
  → remove: pre-decoke tail at run end (feed winding down)
  → remove: after last complete cycle
  → save output/{FURNACE}_clean.parquet
```

**Output:** `output/{FURNACE}_clean.parquet` — indexed by timestamp,
column `run_id` labels each cracking run, column `phase` = `'clean'` everywhere.

Run per furnace:
```bash
bash run_all_furnaces.sh   # uses papermill to inject FURNACE parameter
```


## 1. Parameters

Change `FURNACE` to run on a different furnace. All other settings are shared.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

import sys

# ── Repo root: always 2 levels up from notebooks/00b_clean/ ───────────────
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, REPO_ROOT)

from olefins_ddf.io_events import get_spark
from olefins_ddf import catalog as cat, features as feat_mod
from olefins_ddf.runs import segment_runs, cracking_mask

# ── Parameters (injected by papermill for batch runs) ─────────────────────
FURNACE    = '1HA'           # change per furnace
START      = '2025-02-05'   # dense tube TC start date
END        = '2026-03-20'
BUCKET_MIN = 30             # minutes — 30 min for feature matrix
# Output goes to output/{FURNACE}/00b_clean/ — PNG and CSV only, no time series
# This path is created in Databricks Workspace; download from there to local Mac
OUTPUT_DIR = os.path.join(REPO_ROOT, 'output', FURNACE, '00b_clean')  # always repo root

# ── Delta catalog for research outputs — stays inside Databricks ─────────
# Format: <unity_catalog>.<schema>
# Ask your admin which catalog you have WRITE access to.
DELTA_CATALOG = 'indorama_corporate_olefins_paas_azure_weu_dev_research.workspaces'

# ── Analysis window thresholds (from notebook_01 calibration on 1HA) ──────
FEED_SETTLED_FRAC = 0.95   # feed_total > 95% run median => feed is up
COT_STD_THRESH_C  = 2.0    # rolling std(COT) < 2°C/4h => COT settled
COT_ROLL_STEPS    = 8      # 8 × 30 min = 4 h
MIN_WARMUP_H      = 12.0   # always skip first 12h (Delta-Delta baseline window)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Furnace : {FURNACE}')
print(f'Window  : {START} → {END}')
print(f'Bucket  : {BUCKET_MIN} min')


## 2. Connect to Databricks & load catalog

> **If cluster is sleeping:** run `databricks clusters start <id>` first.
> Update `DDF_CLUSTER_ID` env var when cluster ID changes.


In [ ]:
spark   = get_spark()   # uses DDF_CLUSTER_ID env var
catalog = cat.build_catalog(spark)   # from cache unless refresh=True

# Ensure research schema exists before any Delta writes
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {DELTA_CATALOG}')
print(f'Schema ready: {DELTA_CATALOG}')

# Quick sanity: how many tube TCs for this furnace?
tube_tags = catalog[catalog['role'] == 'tube_COT']
if 'furnace' in tube_tags.columns:
    tube_tags = tube_tags[tube_tags['furnace'] == FURNACE]
print(f'Catalog loaded: {len(catalog)} total tags')
print(f'tube_COT for {FURNACE}: {len(tube_tags)} tags (expect 192)')


## 3. Load feature matrix

Load from **Delta table** in Databricks if it already exists. Otherwise build from raw historian data (~5–10 min) and save back to Delta.
**No parquet is written to local disk.** All processed data stays in Databricks.


In [ ]:
DELTA_TABLE_FEAT = f'{DELTA_CATALOG}.{FURNACE.lower()}_features'

try:
    print(f'Loading cached feature matrix from Delta: {DELTA_TABLE_FEAT}')
    feat_spark = spark.table(DELTA_TABLE_FEAT)
    feat = feat_spark.toPandas()
    # Restore DatetimeIndex
    if 'timestamp' in feat.columns:
        feat = feat.set_index('timestamp')
        feat.index = pd.to_datetime(feat.index)
    print(f'Loaded {len(feat):,} rows from Delta cache.')
except Exception as e:
    print(f'Delta cache not found ({e}). Building from Databricks (~5–10 min)...')
    feat, _ = feat_mod.build_feature_matrix(
        spark, catalog, FURNACE, start=START, end=END, bucket=BUCKET_MIN
    )
    # Save to Delta — data stays in Databricks, never written to local disk
    sdf = spark.createDataFrame(feat.reset_index().rename(columns={'index': 'timestamp'}))
    (sdf.write
       .format('delta')
       .mode('overwrite')
       .option('overwriteSchema', 'true')
       .saveAsTable(DELTA_TABLE_FEAT))
    print(f'Saved to Delta: {DELTA_TABLE_FEAT}')

print(f'Feature matrix : {feat.shape[0]:,} rows × {feat.shape[1]} columns')
print(f'Time range     : {feat.index.min()} → {feat.index.max()}')
print(f'Columns        : {list(feat.columns)}')


## 4. Detect cracking runs

A run = contiguous cracking period between two decokes.
Decoke detected from **HC feed drop to ~0** (not temperature — decoke does NOT cool the furnace).


In [ ]:
# Build cracking mask from feed_total
feed_col = 'feed_total' if 'feed_total' in feat.columns else None
if feed_col is None:
    # Try summing pass feeds
    feed_cols = [c for c in feat.columns if c.startswith('feed_flow') or c == 'feed_total']
    print(f'feed_total not found; available feed cols: {feed_cols}')
    raise ValueError('Cannot detect runs without feed data')

mask = cracking_mask(feat[feed_col])
runs_all = segment_runs(mask)

# ── Keep only COMPLETE cycles: decoke on BOTH sides ───────────────────────
# Rule: exclude any run that starts within 24h of our data window start
# (no confirmed preceding decoke = incomplete cycle on the left)
# Also exclude the last run if it ends within 24h of our data window end
# (no confirmed following decoke = incomplete cycle on the right)
DATA_START = pd.Timestamp(START)
DATA_END   = pd.Timestamp(END)
MIN_GAP_H  = 24  # hours from data boundary → run is incomplete

runs = [r for r in runs_all
        if (r.start - DATA_START).total_seconds() / 3600 >= MIN_GAP_H
        and (DATA_END - r.end).total_seconds() / 3600 >= MIN_GAP_H]

print(f'All runs detected       : {len(runs_all)}')
print(f'Complete cycles (kept)  : {len(runs)}  (both sides bounded by decoke)')
print(f'Incomplete cycles (cut) : {len(runs_all) - len(runs)}  (no decoke confirmed on one side)')
print()
for r in runs:
    print(f'  Run {r.index:2d}: {r.start.date()} → {r.end.date()}  ({r.length_days:.1f} d)')


## 5. Find analysis window for each run

For each run, find the stable cracking window:
- **Start:** feed > 95% median AND rolling std(COT) < 2°C/4h AND t ≥ run.start + 12h
- **End:** last timestamp where feed > 95% median


In [ ]:
def find_analysis_window(run, feat_df: pd.DataFrame) -> dict:
    """Find stable cracking analysis window inside one run."""
    seg = feat_df.loc[run.start:run.end].copy()
    null = dict(run=run.index, run_start=run.start, run_end=run.end,
                analysis_start=pd.NaT, analysis_end=pd.NaT,
                warmup_hours=np.nan, tail_hours=np.nan,
                length_days=round(run.length_days, 2),
                clean_days=np.nan, settled=False)
    if seg.empty:
        return null

    earliest_start = run.start + pd.Timedelta(hours=MIN_WARMUP_H)

    # Criterion 1: feed at operating level
    run_med_feed = seg['feed_total'][seg['feed_total'] > 0].median()
    feed_ok = seg['feed_total'] > FEED_SETTLED_FRAC * run_med_feed

    # Criterion 2: COT settled (rolling std < threshold)
    cot_col = next((c for c in ['cot', 'cot_ctrl'] if c in seg.columns), None)
    if cot_col:
        cot_std = seg[cot_col].rolling(COT_ROLL_STEPS, min_periods=2).std()
        cot_ok  = cot_std < COT_STD_THRESH_C
    else:
        cot_ok = pd.Series(True, index=seg.index)

    # Criterion 3: minimum time past run start
    time_ok = seg.index >= earliest_start

    # Combined
    settled_mask = feed_ok & cot_ok & time_ok
    settled_idx  = settled_mask[settled_mask].index

    if settled_idx.empty:
        analysis_start = earliest_start
        settled = False
    else:
        analysis_start = settled_idx[0]
        settled = True

    warmup_hours = (analysis_start - run.start).total_seconds() / 3600

    # End: last timestamp where feed is still up
    end_valid_idx = feed_ok[feed_ok].index
    analysis_end  = end_valid_idx[-1] if not end_valid_idx.empty else run.end
    # analysis_end must be after analysis_start
    if analysis_end <= analysis_start:
        analysis_end = run.end

    tail_hours = (run.end - analysis_end).total_seconds() / 3600
    clean_days = (analysis_end - analysis_start).total_seconds() / 86400

    return dict(
        run=run.index, run_start=run.start, run_end=run.end,
        analysis_start=analysis_start, analysis_end=analysis_end,
        warmup_hours=round(warmup_hours, 1), tail_hours=round(tail_hours, 1),
        length_days=round(run.length_days, 2), clean_days=round(clean_days, 2),
        settled=settled,
    )


windows = [find_analysis_window(r, feat) for r in runs]
win_df  = pd.DataFrame(windows)

print(f'Windows computed for {len(win_df)} runs')
print(f'  Settled    : {win_df["settled"].sum()} / {len(win_df)}')
print(f'  Warmup avg : {win_df["warmup_hours"].mean():.1f} h  '
      f'(range {win_df["warmup_hours"].min():.0f}–{win_df["warmup_hours"].max():.0f} h)')
print(f'  Tail avg   : {win_df["tail_hours"].mean():.1f} h')
print(f'  Clean avg  : {win_df["clean_days"].mean():.1f} d')
display(win_df[['run','run_start','run_end','analysis_start','analysis_end',
                'warmup_hours','tail_hours','clean_days','settled']])


## 6. Apply all cuts — build clean time series

Four things removed:
1. **Before first complete cycle** — no known run start
2. **Decoking periods** — feed ≈ 0, past process
3. **Warm-up** — feed stepping up, COT unstable, ΔΔ baseline not yet set
4. **Pre-decoke tail** — feed winding down before stop

Only rows inside a valid analysis window are kept.


In [ ]:
# Build a clean mask over the full time series
clean_mask = pd.Series(False, index=feat.index)
run_id_col = pd.Series(np.nan, index=feat.index)

valid_runs = win_df[win_df['analysis_start'].notna() & win_df['analysis_end'].notna()].copy()

for _, w in valid_runs.iterrows():
    a_start = w['analysis_start']
    a_end   = w['analysis_end']
    if pd.isna(a_start) or pd.isna(a_end) or a_end <= a_start:
        continue
    in_window = (feat.index >= a_start) & (feat.index <= a_end)
    clean_mask[in_window] = True
    run_id_col[in_window] = int(w['run'])

feat_clean = feat[clean_mask].copy()
feat_clean['run_id'] = run_id_col[clean_mask].astype(int)
feat_clean['phase'] = 'clean'

total_rows  = len(feat)
clean_rows  = len(feat_clean)
pct_kept    = 100 * clean_rows / total_rows

print(f'Total rows in feature matrix : {total_rows:,}')
print(f'Rows kept (clean windows)    : {clean_rows:,}  ({pct_kept:.1f}%)')
print(f'Rows removed                 : {total_rows - clean_rows:,}  ({100-pct_kept:.1f}%)')
print(f'Complete runs in clean data  : {feat_clean["run_id"].nunique()}')
print(f'\nClean data span:')
print(f'  From : {feat_clean.index.min()}')
print(f'  To   : {feat_clean.index.max()}')
print(f'  Total clean time: {clean_rows * BUCKET_MIN / 60 / 24:.1f} days')


## 7. Manual exclusions

Add known sensor faults or operational events that survived the automated filter.
Harry's principle: only remove physically impossible events (not real high-ΔΔ periods).

> **1HA known issue:** 1 spike ~120°C in dd_abs_max (Oct/Nov 2025, Run 9) lasted >3h.
> Needs DCS verification before excluding — leave commented until confirmed.


In [ ]:
# ── Manual exclusions — fill in after DCS verification ───────────────────
# Format: (reason, start_timestamp, end_timestamp)
MANUAL_EXCLUDE = [
    # ('Run9_sensor_fault', '2025-10-28', '2025-11-02'),  # ~120°C spike, verify first
]

excluded_count = 0
for reason, t0, t1 in MANUAL_EXCLUDE:
    mask_exc = (feat_clean.index >= t0) & (feat_clean.index <= t1)
    n = mask_exc.sum()
    if 'dd_abs_max' in feat_clean.columns:
        feat_clean.loc[mask_exc, 'dd_abs_max'] = np.nan
    excluded_count += n
    print(f'  Manual exclusion: {n} rows NaN-ed — {reason}')

if not MANUAL_EXCLUDE:
    print('No manual exclusions applied.')
else:
    print(f'Total manually excluded: {excluded_count} rows')


## 8. Save clean data to Databricks Delta table

Clean data is written as a **Delta table** inside Databricks — never to local disk.
Downstream notebooks read from this Delta table via `spark.table()`.

The small run-windows metadata CSV (`run_analysis_windows.csv`) is the **only** file saved locally — it contains ~15 rows of run start/end dates and warmup metadata, not time-series data.


In [ ]:
DELTA_TABLE_CLEAN = f'{DELTA_CATALOG}.{FURNACE.lower()}_clean'

# ── Save clean feature matrix to Delta — stays in Databricks ─────────────
sdf_clean = spark.createDataFrame(
    feat_clean.reset_index().rename(columns={'index': 'timestamp'})
)
(sdf_clean.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(DELTA_TABLE_CLEAN))

size_rows = feat_clean.shape[0]
print(f'Clean data saved to Delta : {DELTA_TABLE_CLEAN}')
print(f'Shape                     : {feat_clean.shape}')
print(f'Columns                   : {list(feat_clean.columns)}')

# ── Run windows: small metadata only (~15 rows) — OK to save locally ─────
win_path = os.path.join(OUTPUT_DIR, 'run_analysis_windows.csv')
win_df.to_csv(win_path, index=False)
print(f'\nRun windows metadata saved locally: {win_path}')
print('(This is metadata only — run start/end/warmup hours, NOT time-series data)')


## 9. Verification plot

Visual check: what was kept (green) vs removed (red/grey) across the full time series.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
fig.suptitle(f'{FURNACE} — Clean extraction summary (green=kept · red=removed)',
             fontsize=12, fontweight='bold')

# ── Panel 1: Feed total ───────────────────────────────────────────────────
ax1 = axes[0]
if 'feed_total' in feat.columns:
    ax1.plot(feat.index, feat['feed_total'], color='#378add', lw=0.6, alpha=0.7,
             label='feed_total (raw)')
    ax1.plot(feat_clean.index, feat_clean['feed_total'], color='#1d9e75', lw=0.8,
             label='feed_total (clean)')
ax1.set_ylabel('Feed total\n(NM³/H)')
ax1.legend(fontsize=8, loc='upper right')

# ── Panel 2: COT ─────────────────────────────────────────────────────────
ax2 = axes[1]
cot_col = next((c for c in ['cot', 'cot_ctrl'] if c in feat.columns), None)
if cot_col:
    ax2.plot(feat.index, feat[cot_col], color='#7f77dd', lw=0.6, alpha=0.7,
             label=f'{cot_col} (raw)')
    ax2.plot(feat_clean.index, feat_clean[cot_col], color='#085041', lw=0.8,
             label=f'{cot_col} (clean)')
ax2.set_ylabel('COT (°C)')
ax2.legend(fontsize=8, loc='upper right')

# ── Panel 3: dd_abs_max ───────────────────────────────────────────────────
ax3 = axes[2]
if 'dd_abs_max' in feat.columns:
    ax3.plot(feat.index, feat['dd_abs_max'], color='#ef9f27', lw=0.6, alpha=0.5,
             label='dd_abs_max (raw)')
    ax3.plot(feat_clean.index, feat_clean['dd_abs_max'], color='#d85a30', lw=0.9,
             label='dd_abs_max (clean)')
    ax3.axhline(45, color='#e24b4a', ls='--', lw=1.0, label='Alarm 45°C')
    ax3.axhline(30, color='#ef9f27', ls='--', lw=1.0, label='Pre-alarm 30°C')
ax3.set_ylabel('dd_abs_max (°C)')
ax3.legend(fontsize=8, loc='upper right')

# ── Shade clean windows ───────────────────────────────────────────────────
for _, w in valid_runs.iterrows():
    for ax in axes:
        # Warm-up (red)
        ax.axvspan(w['run_start'], w['analysis_start'],
                   alpha=0.12, color='#e24b4a', zorder=0)
        # Clean (green)
        ax.axvspan(w['analysis_start'], w['analysis_end'],
                   alpha=0.10, color='#1d9e75', zorder=0)
        # Tail (red)
        ax.axvspan(w['analysis_end'], w['run_end'],
                   alpha=0.12, color='#e24b4a', zorder=0)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, f'{FURNACE}_00b_clean_extraction.png')
plt.savefig(plot_path, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')


## 10. Summary

What this output contains and how to use it in downstream notebooks.


In [ ]:
print('=' * 60)
print(f'CLEAN EXTRACTION SUMMARY — {FURNACE}')
print('=' * 60)
print(f'  Furnace          : {FURNACE}')
print(f'  Raw data window  : {START} → {END}')
print(f'  Total runs found : {len(runs)}')
print(f'  Complete cycles  : {len(valid_runs)}  (bounded by decoke on both sides)')
print(f'  Clean rows kept  : {clean_rows:,}  ({pct_kept:.1f}% of raw)')
print(f'  Total clean time : {clean_rows * BUCKET_MIN / 60 / 24:.1f} days')
print()
print('Data storage (all in Databricks — nothing raw saved locally):')
print(f'  Feature matrix   : {DELTA_TABLE_FEAT}')
print(f'  Clean data       : {DELTA_TABLE_CLEAN}')
print(f'  Run windows CSV  : {win_path}  (metadata only, ~15 rows)')
print(f'  Verification PNG : {os.path.join(OUTPUT_DIR, FURNACE + "_00b_clean_extraction.png")}')
print()
print('How to use in downstream notebooks:')
print(f'  feat_clean = spark.table("{DELTA_TABLE_CLEAN}").toPandas()')
print( '  feat_clean = feat_clean.set_index("timestamp")')
print( '  # All rows are already clean — no further filtering needed')
print( '  # Use run_id column to iterate over individual runs')
print()

# Flag suspicious runs for DCS verification
suspicious = win_df[(win_df['warmup_hours'] > 48) | (win_df['tail_hours'] > 48)]
if len(suspicious):
    print('⚠  Suspicious runs (warmup or tail > 48h) — verify with DCS:')
    for _, s in suspicious.iterrows():
        print(f'   Run {int(s["run"]):2d}: warmup={s["warmup_hours"]:.0f}h, '
              f'tail={s["tail_hours"]:.0f}h  ({s["run_start"].date()} → {s["run_end"].date()})')
